# LUTM-6 · evolve programs

NumPy island GA; choose CUDA or NumPy for physical simulation. Each run starts with fresh random programs unless resuming a checkpoint. The program varies, never the transition table or input encoding. Progress is printed while training.


In [ ]:
from pathlib import Path
import numpy as np

from utils import Dataset, SimulationConfig, program_string, plot_history
from evolution import GAConfig, Trainer, checkpoint_config
from numpy_backend import NumpySimulator

backend = "cuda"  # explicitly "cuda" or "numpy"; no fallback
capacity = 2048  # simulator microbatch size, not total population
resume_path = None  # e.g. Path("runs/20261015T120000Z-abcd1234/checkpoint.npz")


In [ ]:
# Supplied pairs are the only training supervision. Repeated cases keep their weight.
train_inputs = ["0", "1", "01", "110", "00101"]
train_targets = ["1", "0", "10", "001", "11010"]
test_inputs = ["", "10110", "1110001", "01011001"]
test_targets = ["", "01001", "0001110", "10100110"]

train_data = Dataset(train_inputs, train_targets)
test_data = Dataset(test_inputs, test_targets)


In [ ]:
# Structural simulator settings. Both tapes have independent blank padding.
simulation = SimulationConfig(
    max_program_length=128,
    program_left_budget=8, program_right_budget=136,
    data_left_budget=32, data_right_budget=255,
    t_max=20_000,
)

# Inherited research defaults; population/budgets here are a modest demonstration.
ga = GAConfig(
    islands=4, population_per_island=256, alphabet="01#", seed=4,
    elite_fraction=.03,
    cross_island_fraction=.02, cross_island_interval=30,  # short-demo override; default is 1200
    random_immigrant_fraction=.15,
    delta_thresholds=(.05, .20), winner_probabilities=(.53, .58), base_winner_probability=.80,
    p_crossover=.50, crossover_segments=3, crossover_min_length=1, crossover_max_length=20,
    p_insert=.04, p_delete=.06, p_symbol_mutation=.02,
    k_penalty=.50, stop_first_exact=True,
)
additional_generations = 300
print_every, diagnostics_every, checkpoint_every = 10, 10, 100


In [ ]:
# Resume takes its locked configs AND datasets from disk, ignoring the cells above.
if resume_path is not None:
    simulation = checkpoint_config(resume_path)

if backend == "cuda":
    from taichi_backend import initialize_cuda, TaichiSimulator
    initialize_cuda()  # run once per kernel; restart kernel before rerunning this cell
    simulator = TaichiSimulator(simulation, capacity)
elif backend == "numpy":
    simulator = NumpySimulator(simulation, capacity)
else:
    raise ValueError("choose cuda or numpy")

trainer = (Trainer.resume(resume_path, simulator) if resume_path is not None
           else Trainer(simulator, train_data, ga, test_dataset=test_data))
print("run:", trainer.run_id, "generation:", trainer.generation)
print("locked GA:", trainer.config)
print("checkpoint:", trainer.checkpoint_path)


In [ ]:
# Interrupt this cell to save the last fully evaluated generation.
# A currently running CUDA kernel finishes before Python handles Ctrl+C.
trainer.run(additional_generations, print_every=print_every,
            diagnostics_every=diagnostics_every, checkpoint_every=checkpoint_every)


In [ ]:
plot_history(trainer.history);


## Inspect current island champions and the best-ever archive

Champion resource means include only successful halted input cases. Undefined values are `NaN`, not zero. The best-ever archive is reporting state; it is not continually reinserted into the population.


In [ ]:
from utils import TIME, P_SPACE, X_SPACE

evaluation = trainer.evaluation
flat = trainer.population.reshape(-1, trainer.population.shape[-1])
lengths = trainer.lengths.ravel()
for island, index in enumerate(trainer.champions()):
    print(f"island={island} fitness={evaluation.fitness[index]:.6f} accuracy={evaluation.accuracy[index]:.4f} "
          f"invalid={evaluation.invalid_fraction[index]:.4f} T={evaluation.resource_mean(TIME)[index]:.1f} "
          f"P_span={evaluation.resource_mean(P_SPACE)[index]:.1f} X_span={evaluation.resource_mean(X_SPACE)[index]:.1f}")
    print(program_string(flat[index], lengths[index]))

best_program = program_string(trainer.best["program"], trainer.best["length"])
print("best ever:", best_program)
print({k: v for k, v in trainer.best.items() if k != "program"})


## Held-out evaluation

This evaluation neither feeds selection nor consumes the training RNG. A perfect finite training fit does not imply correctness at arbitrary lengths.


In [ ]:
heldout = trainer.evaluate_best()
print("test accuracy:", heldout.accuracy[0], "invalid:", heldout.invalid_fraction[0], "exact:", heldout.exact[0])
values = trainer.best["program"][None, :]
lengths = np.array([trainer.best["length"]], dtype=np.int32)
for x, target in zip(trainer.test_dataset.inputs, trainer.test_dataset.targets):
    outcome = simulator.simulate(values, lengths, x)
    print(repr(x), "->", outcome.row(), "target:", repr(target))


In [ ]:
# Save/export this run. Re-running trainer.run(...) continues from current state.
checkpoint = trainer.save()
program_file = checkpoint.parent / "best_program.txt"
program_file.write_text(best_program + "\n", encoding="ascii")
print("saved:", checkpoint)
print("literal:", program_file)
